# Manejo de Errores en Rust

En este cuaderno veremos:
- `panic!` (errores no recuperables)
- Manejo de errores con `match`
- Lectura de archivos con `Result`

Basado en los ejemplos de la especialización (Coursera).

## 1) Panicking: `panic!`

Usamos `panic!` cuando no podemos o no queremos recuperarnos del error.

In [6]:
fn loop_and_panic(numbers: Vec<i32>) {
    for num in numbers {
        if num < 0 {
            panic!("Negative number found!");
        }
        println!("Number: {}", num);
    }
}

fn main() {
    loop_and_panic(vec![1, 2, 3, 4, -5]);
}
main()


Number: 1
Number: 2
Number: 3
Number: 4



thread '<unnamed>' panicked at src/lib.rs:11:13:
Negative number found!
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/std/src/panicking.rs:697:5
   1: core::panicking::panic_fmt
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/core/src/panicking.rs:75:14
   2: ctx::main
   3: std::panic::catch_unwind
   4: run_user_code_5
   5: evcxr::runtime::Runtime::run_loop
   6: evcxr::runtime::runtime_hook
   7: evcxr_jupyter::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


## 2) Manejo de errores con `match` + `Result`

`File::open` devuelve `Result<File, std::io::Error>`.
Con `match` distinguimos éxito (`Ok`) de error (`Err`).

In [11]:
use std::fs::File;
use std::io::{BufRead, BufReader};

fn main() {
    let file = File::open("README.txt");
    let file = match file {
        Ok(file) => file,
        Err(error) => {
            match error.kind() {
                std::io::ErrorKind::NotFound => {
                    panic!("File not found: {}", error)
                }
                _ => {
                    panic!("Error opening file: {}", error)
                }
            }
        }
    };

    let reader = BufReader::new(file);
    for line in reader.lines() {
        match line {
            Ok(line) => println!("{}", line),
            Err(error) => {
                panic!("Error reading line: {}", error)
            }
        }
    }
}
main()



thread '<unnamed>' panicked at src/lib.rs:10:21:
File not found: No such file or directory (os error 2)
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/std/src/panicking.rs:697:5
   1: core::panicking::panic_fmt
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/core/src/panicking.rs:75:14
   2: <unknown>
   3: <unknown>
   4: <unknown>
   5: evcxr::runtime::Runtime::run_loop
   6: evcxr::runtime::runtime_hook
   7: evcxr_jupyter::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


## 3) Resumen

- `panic!` corta el programa.
- `Result<T, E>` permite manejar errores de forma explícita.
- `match` es la base para tratar `Ok`/`Err` de forma segura.

Siguiente paso recomendado: `unwrap`, `expect`, `Option<T>` y operador `?`.

## 4) `unwrap` y `expect`

Ambos métodos extraen el valor de `Ok` (o `Some`).
Si hay error (`Err` o `None`), el programa hace `panic!`.

- `unwrap()`: mensaje genérico.
- `expect("mensaje")`: mensaje personalizado, ideal para depurar.

In [ ]:
fn dividir(a: i32, b: i32) -> Result<i32, String> {
    if b == 0 {
        Err("No se puede dividir por cero".to_string())
    } else {
        Ok(a / b)
    }
}

println!("10 / 2 con unwrap: {}", dividir(10, 2).unwrap());

match dividir(10, 0) {
    Ok(v) => println!("Resultado: {}", v),
    Err(e) => println!("Error controlado: {}", e),
}

// Si quieres mostrar panic con mensaje personalizado, descomenta:
// let _ = dividir(10, 0).expect("Fallo al dividir en ejemplo con expect");

10 / 2 con unwrap: 5
Error controlado: No se puede dividir por cero



thread '<unnamed>' panicked at src/lib.rs:60:24:
Fallo al dividir en ejemplo con expect: "No se puede dividir por cero"
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/std/src/panicking.rs:697:5
   1: core::panicking::panic_fmt
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/core/src/panicking.rs:75:14
   2: core::result::unwrap_failed
             at /rustc/29483883eed69d5fb4db01964cdf2af4d86e9cb2/library/core/src/result.rs:1761:5
   3: <unknown>
   4: <unknown>
   5: evcxr::runtime::Runtime::run_loop
   6: evcxr::runtime::runtime_hook
   7: evcxr_jupyter::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.


## 5) `Option<T>` para valores opcionales

`Option<T>` se usa cuando un valor puede existir o no.
No hay `null`: usamos `Some(valor)` o `None`.

In [12]:
fn buscar_usuario(id: i32) -> Option<&'static str> {
    match id {
        1 => Some("Ana"),
        2 => Some("Luis"),
        _ => None,
    }
}

match buscar_usuario(1) {
    Some(nombre) => println!("Usuario encontrado: {}", nombre),
    None => println!("No se encontró el usuario"),
}

match buscar_usuario(99) {
    Some(nombre) => println!("Usuario encontrado: {}", nombre),
    None => println!("No se encontró el usuario"),
}

Usuario encontrado: Ana
No se encontró el usuario


()

## 6) Propagación de errores con `?`

El operador `?` simplifica funciones que devuelven `Result`.
Si ocurre un `Err`, se retorna automáticamente al llamador.

In [13]:
fn parsear_entero(texto: &str) -> Result<i32, std::num::ParseIntError> {
    let numero = texto.parse::<i32>()?;
    Ok(numero)
}

match parsear_entero("42") {
    Ok(n) => println!("Parseo correcto: {}", n),
    Err(e) => println!("Error al parsear: {}", e),
}

match parsear_entero("hola") {
    Ok(n) => println!("Parseo correcto: {}", n),
    Err(e) => println!("Error al parsear: {}", e),
}

Parseo correcto: 42
Error al parsear: invalid digit found in string


()

## 7) Práctica en clase (10-15 min)

1. Crea una función `safe_div(a, b) -> Result<i32, String>`.
2. Si `b == 0`, devuelve `Err`.
3. Si no, devuelve `Ok(a / b)`.
4. En el bloque principal, prueba con `(20, 4)` y `(20, 0)` usando `match`.

Objetivo: decidir cuándo usar `panic!` y cuándo `Result`.

In [ ]:
fn safe_div(a: i32, b: i32) -> Result<i32, String> {
    if b == 0 {
        Err("No se puede dividir por cero".to_string())
    } else {
        Ok(a / b)
    }
}

fn main() {
    match safe_div(20, 4) {
        Ok(v) => println!("20 / 4 = {}", v),
        Err(e) => println!("Error: {}", e),
    }

    match safe_div(20, 0) {
        Ok(v) => println!("20 / 0 = {}", v),
        Err(e) => println!("Error: {}", e),
    }
}
main()

20 / 4 = 5
Error: No se puede dividir por cero

## Cuándo Usar panic! vs Result

| Escenario | Usar | Motivo |
|---|---|---|
| Entrada de usuario inválida (por ejemplo, número mal escrito) | Result | Es un error esperable y recuperable. |
| Archivo no encontrado o sin permisos | Result | El programa puede informar el error y continuar o reintentar. |
| División por cero en lógica de negocio | Result | Puede ocurrir en ejecución normal, no debe tirar toda la app. |
| Parseo de texto a número | Result | Fallo común en runtime; se maneja con mensajes claros. |
| Búsqueda de dato que puede no existir | Option o Result | La ausencia es parte normal del flujo. |
| API de librería para terceros | Result | Le das control al consumidor para decidir cómo tratar el fallo. |
| Estado imposible por contrato interno roto | panic! | Indica bug de programación; conviene fallar rápido. |
| Invariante crítica violada (nunca debería pasar) | panic! | Señala error grave que debe corregirse en código. |
| Prototipo o script rápido de aprendizaje | panic! con cuidado | Simplifica, pero no es ideal para producción. |
| Tests donde algo debe cumplirse sí o sí | panic! o expect | Hace visible de inmediato una condición incumplida. |



| Regla rápida | Decisión |
|---|---|
| Si puede fallar por usuario, red o disco | Result |
| Si solo puede pasar por bug del programador | panic! |

## ErrorKind predefinidos (std::io::ErrorKind)
En tu versión de Rust (1.89.0) son **42** variantes.



| Grupo | Variantes |
|---|---|
| Archivo/Sistema de archivos | NotFound, AlreadyExists, NotADirectory, IsADirectory, DirectoryNotEmpty, ReadOnlyFilesystem, FilesystemLoop, StaleNetworkFileHandle, NotSeekable, StorageFull, QuotaExceeded, FileTooLarge, TooManyLinks, CrossesDevices, InvalidFilename, ExecutableFileBusy |
| Permisos/Entrada | PermissionDenied, InvalidInput, InvalidData, ArgumentListTooLong |
| Red/Conectividad | ConnectionRefused, ConnectionReset, HostUnreachable, NetworkUnreachable, ConnectionAborted, NotConnected, AddrInUse, AddrNotAvailable, NetworkDown |
| Estado de operación | WouldBlock, TimedOut, WriteZero, Interrupted, InProgress, Deadlock, ResourceBusy |
| Plataforma/Capacidad | Unsupported, OutOfMemory |
| Flujo genérico | BrokenPipe, UnexpectedEof, Other, Uncategorized |

Nota docente corta:
- El número puede cambiar con versiones futuras.
- Para “lo demás”, suele usarse `_` en `match` como fallback.


## Cómo crear errores propios de dominio (personalizados)


### Opción 1: Manual (sin dependencias)


In [15]:
use std::error::Error;
use std::fmt;

#[derive(Debug)]
enum PagoError {
    MontoInvalido(f64),
    SaldoInsuficiente { saldo: f64, intento: f64 },
    MetodoNoSoportado(String),
}

impl fmt::Display for PagoError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            PagoError::MontoInvalido(m) => write!(f, "Monto inválido: {}", m),
            PagoError::SaldoInsuficiente { saldo, intento } => {
                write!(f, "Saldo insuficiente. Saldo: {}, intento: {}", saldo, intento)
            }
            PagoError::MetodoNoSoportado(metodo) => {
                write!(f, "Método de pago no soportado: {}", metodo)
            }
        }
    }
}

impl Error for PagoError {}

fn procesar_pago(monto: f64, saldo: f64, metodo: &str) -> Result<(), PagoError> {
    if monto <= 0.0 {
        return Err(PagoError::MontoInvalido(monto));
    }
    if monto > saldo {
        return Err(PagoError::SaldoInsuficiente { saldo, intento: monto });
    }
    if metodo != "tarjeta" && metodo != "transferencia" {
        return Err(PagoError::MetodoNoSoportado(metodo.to_string()));
    }
    Ok(())
}

fn main() {
    match procesar_pago(150.0, 100.0, "tarjeta") {
        Ok(_) => println!("Pago aprobado"),
        Err(e) => println!("Error de dominio: {}", e),
    }
}
main()


Error de dominio: Saldo insuficiente. Saldo: 100, intento: 150


()

### Opción 2: Recomendada en proyectos reales (`thiserror`)

Más limpia para mantener.

In [ ]:
use thiserror::Error;

#[derive(Debug, Error)]
enum PagoError {
    #[error("Monto inválido: {0}")]
    MontoInvalido(f64),

    #[error("Saldo insuficiente. Saldo: {saldo}, intento: {intento}")]
    SaldoInsuficiente { saldo: f64, intento: f64 },

    #[error("Método no soportado: {0}")]
    MetodoNoSoportado(String),
}



## Regla práctica para tus alumnos


- `ErrorKind` (std) = errores técnicos de IO/sistema.
- Error de dominio propio = reglas de negocio (pagos, pedidos, cupos, etc.).
- Combinalos en capas: infraestructura devuelve `io::Error`, dominio devuelve tu `enum` propio.

Si querés, te preparo una celda markdown + una celda de código para pegar directo en tu notebook con este contenido.